# 01 — Explore ChickenVerse (ChickenDet)

**Phase 1 (Exploration & Learning)** notebook, per [`plan.md`](../plan.md) and [`constitution.md`](../constitution.md) Principle II — run this on **Google Colab** (free T4 GPU) for the actual work, not locally.

**Goal for this notebook:** get familiar with the ChickenDet dataset before any modeling — load the COCO annotations (boxes + masks), visualize samples, and look at instance-density distribution across the train/val/test splits (test is deliberately denser, ~50.5 avg vs. ~23.5 avg overall — worth seeing that visually before it shows up as a training/eval difficulty later).

**Not in scope here:** any model training — that starts in the next notebook once this one has done its job.

---

**When done, capture what you learned in the "Notes" section at the bottom** — per constitution Principle II, that's the actual design input for Phase 2+, not the notebook code itself.

## Setup (Colab)

In [ ]:
# Run on Colab (Runtime > Change runtime type > T4 GPU)
# %pip install pycocotools matplotlib numpy pillow

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Download ChickenVerse (ChickenDet split)

Source: https://github.com/amirivojdan/ChickenVerse, download via [Zenodo](https://zenodo.org/records/20672799). License: CC BY-NC-SA 4.0 — non-commercial, attribution, share-alike (see `constitution.md` Principle VI).

In [ ]:
# ChickenDet (train/val/test images + COCO annotation JSONs) downloaded once and kept on
# Drive so it survives across sessions without re-downloading every time.
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/poultry_monitoring/data/ChickenDet"

## Import Libraries

**pycoco** -> For annotations and dataset handling

**PIL** -> For basic image handling

**matplotlib** -> For charts and image display


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from pycocotools.coco import COCO

np.random.seed(42)  # reproducible sample picks below

## Load COCO annotations (boxes + masks)

Both bounding boxes and pixel-level segmentation masks live in the same COCO-format annotation file per split (SAM2-assisted labeling via CVAT).

In [ ]:
# Point at the actual annotation file paths once DATA_DIR is real
coco_train = COCO(f"{DATA_DIR}/annotations/instances_Train.json")
coco_val = COCO(f"{DATA_DIR}/annotations/instances_Validation.json")
coco_test = COCO(f"{DATA_DIR}/annotations/instances_Test.json")

In [ ]:
# Display Categories
cats = coco_train.loadCats(coco_train.getCatIds())
nms = [cat["name"] for cat in cats]
print(f"Categories: {', '.join(nms)}")

## Visualize samples (boxes + masks overlaid)

Pull a handful of images per split and overlay both bounding boxes and masks — sanity-check that annotations line up with what's actually in the image, and get a feel for occlusion/density before writing any augmentation code.

In [ ]:
# Function to visualize image
def visualize_image(coco, img_id, img_dir, show_masks=False, show_boxes=False, img_title="IMAGE"):
    """Display one image with its COCO boxes and/or masks overlaid.

    Args:
        coco: A loaded `pycocotools.coco.COCO` instance for the split `img_id` belongs to.
        img_id: COCO image ID to display.
        img_dir: Directory containing the image files for this split.
        show_masks: Overlay segmentation masks.
        show_boxes: Overlay bounding boxes.
        img_title: Prefix for the plot title.
    """
    img_info = coco.loadImgs(img_id)[0]
    img_path = f"{img_dir}/{img_info['file_name']}"

    # Open the image first
    with Image.open(img_path) as img:
        plt.figure(figsize=(10, 10))  # Create a new figure for each image
        plt.imshow(img)
        plt.title(f"{img_title} ID: {img_id}")
        plt.axis("off")

        if show_boxes or show_masks:
            # Get all annotations for the current image
            ann_ids = coco.getAnnIds(imgIds=img_id)
            anns = coco.loadAnns(ann_ids)

            # Overlay bounding boxes first if requested
            if show_boxes:
                coco.showAnns(anns, draw_bbox=True)
            # Overlay masks if requested
            if show_masks:
                coco.showAnns(anns, draw_bbox=False)

            # Drop refs to the per-image annotation list so it's freed now rather
            # than lingering until the next loop iteration — matters here since a
            # dense image can carry 100+ polygon masks.
            del anns
            del ann_ids

        # Display the plot with image and annotations
        plt.show()

In [ ]:
# Visualize random train, validation and test images
train_img_ids = coco_train.getImgIds()
rand_img_ind = np.random.randint(0, len(train_img_ids))
train_data_dir = f"{DATA_DIR}/images/Train"
visualize_image(
    coco_train,
    train_img_ids[rand_img_ind],
    train_data_dir,
    show_masks=True,
    show_boxes=False,
    img_title="Train Image",
)

# Visualize random validation
val_img_ids = coco_val.getImgIds()
rand_img_ind = np.random.randint(0, len(val_img_ids))
val_data_dir = f"{DATA_DIR}/images/Validation"
visualize_image(
    coco_val,
    coco_val.getImgIds()[rand_img_ind],
    val_data_dir,
    show_masks=False,
    show_boxes=True,
    img_title="Validation Image",
)

# Visualize random test
test_img_ids = coco_test.getImgIds()
rand_img_ind = np.random.randint(0, len(test_img_ids))
test_data_dir = f"{DATA_DIR}/images/Test"
visualize_image(
    coco_test,
    coco_test.getImgIds()[rand_img_ind],
    test_data_dir,
    show_masks=False,
    show_boxes=True,
    img_title="Test Image",
)

# Flush plot memmory
plt.close("all")

## Instance-density distribution across splits

Per the dataset README: train ~23.5 avg instances/image, test ~50.5 avg (deliberately denser — a stress test). Plot the actual per-image instance-count distribution for each split and confirm this before it becomes a training/eval surprise later.

In [ ]:
# For each split, compute instances-per-image, then plot all three splits together
# with a shared x-axis so the density shift is directly visible rather than
# inferred from separately-scaled plots.

# Fixed categorical colors (dataviz skill palette, slots 1-3) — one hue per split,
# same assignment used consistently anywhere splits are compared going forward.
SPLIT_COLORS = {"Train": "#2a78d6", "Val": "#eb6834", "Test": "#1baf7a"}


def instances_per_image(coco):
    """Return an array with the annotation count for each image in the split."""
    img_ids = coco.getImgIds()
    return np.array([len(coco.getAnnIds(imgIds=img_id)) for img_id in img_ids])

In [ ]:
splits = {"Train": coco_train, "Val": coco_val, "Test": coco_test}
counts = {name: instances_per_image(coco) for name, coco in splits.items()}

for name, arr in counts.items():
    print(
        f"{name}: n_images={len(arr)}, mean={arr.mean():.1f}, "
        f"median={np.median(arr):.1f}, max={arr.max()}"
    )

# Shared bin edges across all three splits so bars are directly comparable —
# separately-binned histograms would silently hide the train/test density shift.
max_count = max(arr.max() for arr in counts.values())
bins = np.linspace(0, max_count, 21)

fig, axes = plt.subplots(len(splits), 1, figsize=(8, 8), sharex=True)
for ax, (name, arr) in zip(axes, counts.items()):
    ax.hist(arr, bins=bins, color=SPLIT_COLORS[name], edgecolor="white", linewidth=0.5)
    ax.axvline(
        arr.mean(),
        color="#0b0b0b",
        linestyle="--",
        linewidth=1,
        label=f"mean = {arr.mean():.1f}",
    )
    ax.set_ylabel("Frequency")
    ax.set_title(f"{name}  (n={len(arr)} images)", loc="left", fontsize=10)
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

axes[-1].set_xlabel("Instances per image")
fig.suptitle("Instance density per image, by split", y=0.995)
fig.tight_layout()
plt.show()

## Notes — what I learned

- **Dataset Sizes & Splits**:
    - **Train**: 2,495 images.
    - **Validation**: 587 images.
    - **Test**: 125 images.
    - The data follows a standard structure with images divided into subdirectories (`Train`, `Validation`, `Test`) and separate JSON annotation files for each.

- **COCO Utilities for Dataset Usage**:
    - `COCO(path)`: Initializes the API and indexes the JSON annotations.
    - `coco.getImgIds()`: Returns a list of all image identifiers in the split.
    - `coco.loadImgs(ids)`: Returns metadata (filename, height, width) for specified images.
    - `coco.getAnnIds(imgIds=id)`: Fetches annotation IDs belonging to a specific image.
    - `coco.showAnns(anns, draw_bbox=True/False)`: Directly overlays segmentation masks and bounding boxes onto the active Matplotlib plot.

- **Split Density Differences (Histograms)**:
    - **Train/Val**: Exhibit similar densities (~22-23 instances per image).
    - **Test**: Significant shift to higher density (~50.5 instances per image).
    - **Surprising Maxima**: One training image contains 106 instances, which might be a challenging outlier for the model.
    - Plotting all three splits on a shared x-axis makes the train/val → test density shift immediately visible, instead of something you'd only notice once it shows up as a harder eval later.

- **Data Directory Structure**:
    - Root: `DATA_DIR`
    - Images: `{DATA_DIR}/images/{Train|Validation|Test}/`
    - Annotations: `{DATA_DIR}/annotations/instances_{Train|Validation|Test}.json`

- **Implementation Learning**:
    - To avoid RAM buildup in Colab when visualizing many images, it's worth explicitly calling `plt.close('all')` (or `plt.close(fig)`) to clear the figure buffer from memory after each batch of plots.
    - Delete unused heavy variables (annotation lists, image handles) as soon as a loop iteration is done with them, rather than letting them live until the next iteration — matters here since some images carry 100+ polygon masks.